# Cours interactif - Comprendre `train_plantar_model.py`

Objectif : comprendre pas a pas comment on entraine un modele de reconnaissance d'activites avec les donnees plantaires.

Ce notebook reprend le protocole utilise dans `train_plantar_model.py` :

1. Lire `Plantar_activity_reference/` et `Events/`.
2. Aligner les signaux de semelles avec les annotations temporelles.
3. Transformer un segment temporel en vecteur de features.
4. Faire un split unique `Train / Validation / Test`.
5. Entrainement par `epochs`, `batch_size` et `iterations`.
6. Detecter l'overfitting avec `train_acc`, `val_acc` et le test final.

Le but n'est pas seulement d'obtenir une accuracy : le but est de comprendre pourquoi le score est credible, et ou sont les limites.

## 0. Avant de commencer

Le notebook suppose que tu es dans le dossier du TD :

`/Users/maxence/Documents/IMT/an2/data/TD1`

Si ton kernel Jupyter n'utilise pas le `.venv`, il faudra peut-etre installer les dependances avec la cellule suivante.

In [ ]:
# A executer seulement si une importation echoue plus bas.
# Dans Jupyter, %pip installe les paquets dans le kernel courant.
# %pip install pandas numpy scikit-learn joblib matplotlib

In [ ]:
import os
import sys
from pathlib import Path
from types import SimpleNamespace

ROOT = Path.cwd()
if not (ROOT / 'Events').exists() and (ROOT.parent / 'Events').exists():
    ROOT = ROOT.parent

os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib-cache'))
scripts_dir = ROOT / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Dossier courant :', ROOT)
print('Python du kernel :', sys.executable)
print('Le dossier Plantar_activity_reference existe :', (ROOT / 'Plantar_activity_reference').exists())
print('Le dossier Events existe :', (ROOT / 'Events').exists())

## 1. Comprendre les fichiers

Le dataset est organise par sujet et sequence.

- `Plantar_activity_reference/S01/Sequence_01/insoles.csv` contient les signaux des semelles : pression, acceleration, gyroscope, force totale, centre de pression.
- `Events/S01/Sequence_01/classif.csv` contient les annotations : nom de l'action, classe, debut et fin temporels.

Une ligne de `insoles.csv` ne contient pas directement le label de l'action. On doit donc utiliser les timestamps de `classif.csv` pour savoir quelle action correspond a chaque intervalle.

In [ ]:
insoles_path = ROOT / 'Plantar_activity_reference' / 'S01' / 'Sequence_01' / 'insoles.csv'
events_path = ROOT / 'Events' / 'S01' / 'Sequence_01' / 'classif.csv'

insoles = pd.read_csv(insoles_path, sep=';')
events = pd.read_csv(events_path, sep=';')

print('insoles shape :', insoles.shape)
print('events shape  :', events.shape)
display(insoles.head())
display(events.head())

## 2. Le principe de l'alignement temporel

Dans `classif.csv`, une action est definie par :

- `Timestamp Start`
- `Timestamp End`
- `Name`

Exemple : si une action commence a 8.19s et finit a 13.09s, alors les lignes de `insoles.csv` dont `Time` est entre 8.19 et 13.09 appartiennent a cette action.

Le script ne garde pas une ligne brute comme exemple d'apprentissage. Il resume tout le segment avec des statistiques : moyenne, ecart-type, min, max, quantiles, RMS, deltas, etc.

In [ ]:
first_event = events.iloc[0]
start = first_event['Timestamp Start']
end = first_event['Timestamp End']
label = first_event['Name']

segment = insoles[(insoles['Time'] >= start) & (insoles['Time'] <= end)].copy()

print('Action :', label)
print('Debut / fin :', start, '->', end, 'secondes')
print('Nombre de lignes capteur dans ce segment :', len(segment))

segment[['Time', 'left total force[N]', 'right total force[N]']].head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(segment['Time'], segment['left total force[N]'], label='force gauche')
plt.plot(segment['Time'], segment['right total force[N]'], label='force droite')
plt.title(f'Exemple de segment annote : {label}')
plt.xlabel('Temps [s]')
plt.ylabel('Force [N]')
plt.legend()
plt.grid(True)
plt.show()

## 3. Reutiliser les fonctions du script

Pour que ce notebook reste coherent avec le vrai entrainement, on importe les fonctions depuis `train_plantar_model.py`.

Important : on ne recopie pas toute la logique dans le notebook. Le notebook sert de cours et d'experimentation, le script reste la source principale.

In [ ]:
import train_plantar_model as tpm

print('Fonctions disponibles :')
for name in ['find_pairs', 'build_dataset', 'split_train_val_test', 'train_mlp_with_validation', 'diagnose_overfitting']:
    print('-', name, hasattr(tpm, name))

## 4. Construire un petit dataset d'apprentissage

Pour apprendre sans attendre trop longtemps, on commence avec `max_files=8`.

Cela ne donnera pas la meilleure accuracy, mais c'est parfait pour comprendre le pipeline. Le script complet utilise les 319 fichiers apparies.

In [ ]:
def make_args(max_files=8, epochs=8, batch_size=32, output='outputs/notebook_demo_model.joblib'):
    return SimpleNamespace(
        plantar_root=ROOT / 'Plantar_activity_reference',
        events_root=ROOT / 'Events',
        output=ROOT / output,
        mode='event',
        window_seconds=2.0,
        stride_seconds=1.0,
        min_samples=20,
        max_samples_per_event=12,
        split='random',
        model='mlp',
        n_estimators=500,
        n_neighbors=5,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=0.001,
        alpha=0.0001,
        hidden_layers='256,128',
        patience=12,
        overfit_gap=0.10,
        val_drop=0.03,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1,
        max_files=max_files,
        val_size=0.15,
        test_size=0.15,
        strict=False,
        no_save=False,
    )

args = make_args(max_files=8, epochs=8, batch_size=32)
X, y, metas, feature_names = tpm.build_dataset(args)

print('X shape :', X.shape)
print('y shape :', y.shape)
print('Nombre de features :', len(feature_names))
print('Nombre de classes :', len(set(y)))
print('Exemples de features :')
feature_names[:10]

## 5. Train / Validation / Test

Le cours insiste sur ce point :

- `Train` : le modele apprend.
- `Validation` : on surveille l'entrainement et on choisit le meilleur epoch.
- `Test` : evaluation finale, une seule fois, apres l'entrainement.

Le scaler et l'imputation sont appris uniquement sur le train. Cela evite de donner indirectement des informations de validation/test au modele.

In [ ]:
(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    train_idx, val_idx, test_idx,
) = tpm.split_train_val_test(
    X, y, metas,
    split=args.split,
    val_size=args.val_size,
    test_size=args.test_size,
    random_state=args.random_state,
)

print('Train :', X_train.shape, y_train.shape)
print('Val   :', X_val.shape, y_val.shape)
print('Test  :', X_test.shape, y_test.shape)

tpm.validate_split(y_train, y_val, y_test, metas, train_idx, val_idx, test_idx, args.split, strict=False)

## 6. Epoch, batch et iteration

Definitions simples :

- Un `batch` est un petit paquet d'exemples envoye au modele.
- Une `iteration` correspond au traitement d'un batch.
- Un `epoch` signifie que le modele a vu tout le train une fois.

Exemple : si on a 179 exemples train et `batch_size=32`, alors il faut environ 6 iterations pour faire 1 epoch.

In [ ]:
n_train = len(y_train)
iterations_par_epoch = int(np.ceil(n_train / args.batch_size))

print('Nombre exemples train :', n_train)
print('Batch size :', args.batch_size)
print('Iterations par epoch :', iterations_par_epoch)

## 7. Entrainement MLP avec validation

On lance maintenant un petit entrainement. Observe la difference entre `train_acc` et `val_acc`.

- Si les deux montent ensemble : bon signe.
- Si `train_acc` monte tres haut mais `val_acc` reste basse : overfitting.
- Si les deux restent basses : underfitting ou features insuffisantes.

Sur un petit sous-ensemble, l'overfitting est normal : il y a trop peu de donnees.

In [ ]:
model, history = tpm.train_mlp_with_validation(X_train, X_val, y_train, y_val, args)
history_df = pd.DataFrame(history)
history_df

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_df['epoch'], history_df['train_accuracy'], marker='o', label='train accuracy')
plt.plot(history_df['epoch'], history_df['val_accuracy'], marker='o', label='validation accuracy')
plt.title('Suivi de l apprentissage')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

## 8. Evaluation finale sur le test

Le test ne sert pas a choisir le modele. Il sert uniquement a mesurer la performance finale du modele selectionne via validation.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
test_balanced_acc = balanced_accuracy_score(y_test, y_pred)

print('Test accuracy :', round(test_acc, 3))
print('Balanced test accuracy :', round(test_balanced_acc, 3))
print(classification_report(y_test, y_pred, zero_division=0))

## 9. Diagnostic automatique de l'overfitting

Le script considere qu'il y a un risque d'overfitting si l'ecart `train_acc - val_acc` devient trop grand.

Par defaut, le seuil est `0.10`, c'est-a-dire 10 points d'accuracy.

In [ ]:
diagnostic = tpm.diagnose_overfitting(
    history,
    test_accuracy=test_acc,
    test_balanced_accuracy=test_balanced_acc,
    overfit_gap=args.overfit_gap,
    val_drop_threshold=args.val_drop,
)

tpm.print_diagnostic(diagnostic)
diagnostic

## 10. Lire le resultat complet deja entraine

Le script complet a deja ete lance sur tous les fichiers. Il a genere :

- `outputs/plantar_activity_model.joblib`
- `outputs/plantar_activity_model_history.csv`
- `outputs/plantar_activity_model_metrics.json`
- `outputs/plantar_activity_model_test_predictions.csv`

On peut les lire pour analyser l'entrainement complet sans le relancer.

In [ ]:
import json

history_path = ROOT / 'outputs' / 'plantar_activity_model_history.csv'
metrics_path = ROOT / 'outputs' / 'plantar_activity_model_metrics.json'

full_history = pd.read_csv(history_path)
metrics = json.loads(metrics_path.read_text())

print('Accuracy test complete :', round(metrics['accuracy'], 3))
print('Balanced accuracy complete :', round(metrics['balanced_accuracy'], 3))
print('Diagnostic :', metrics['diagnostic']['status'])
print('Best epoch :', metrics['diagnostic']['best_epoch'])
print('Best val accuracy :', round(metrics['diagnostic']['best_val_accuracy'], 3))
display(full_history.head())
display(full_history.tail())

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(full_history['epoch'], full_history['train_accuracy'], label='train accuracy')
plt.plot(full_history['epoch'], full_history['val_accuracy'], label='validation accuracy')
plt.axvline(metrics['diagnostic']['best_epoch'], color='red', linestyle='--', label='best epoch')
plt.title('Entrainement complet : train vs validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(full_history['epoch'], full_history['loss'], label='loss')
plt.title('Loss pendant l entrainement complet')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 11. Relancer l'entrainement complet

La cellule suivante execute le script complet depuis le notebook.

Elle peut prendre un peu de temps. Le score attendu est autour de `78%`, donc proche de l'objectif `~80%`.

In [ ]:
# Decommente pour relancer le script complet depuis le notebook.
# %run {ROOT / 'scripts' / 'train_plantar_model.py'}

## 12. Exercices pour maitriser

Essaie de modifier ces parametres dans `make_args()` puis relance les cellules :

1. Reduire le modele : `hidden_layers='128,64'`. Est-ce que l'overfitting baisse ?
2. Augmenter la regularisation : `alpha=0.001`. Est-ce que `train_acc - val_acc` diminue ?
3. Changer `max_files=20`. Est-ce que la validation devient plus stable ?
4. Dans le script complet, tester : `--split subject`. Est-ce que le modele generalise a des sujets jamais vus ?

Regle importante : ne choisis jamais ton modele final avec le test. Tu choisis avec la validation, puis tu annonces le test une seule fois.